In [2]:
url = 'https://anaconda.org/conda-forge/libta-lib/0.4.0/download/linux-64/libta-lib-0.4.0-h166bdaf_1.tar.bz2'
!curl -L $url | tar xj -C /usr/lib/x86_64-linux-gnu/ lib --strip-components=1
url = 'https://anaconda.org/conda-forge/ta-lib/0.4.19/download/linux-64/ta-lib-0.4.19-py310hde88566_4.tar.bz2'
!curl -L $url | tar xj -C /usr/local/lib/python3.10/dist-packages/ lib/python3.10/site-packages/talib --strip-components=3
import talib

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  4472    0  4472    0     0  17876      0 --:--:-- --:--:-- --:--:-- 17888
100  517k  100  517k    0     0   864k      0 --:--:-- --:--:-- --:--:-- 4750k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  4484    0  4484    0     0  18883      0 --:--:-- --:--:-- --:--:-- 18919
100  392k  100  392k    0     0   741k      0 --:--:-- --:--:-- --:--:--  741k


In [3]:
!pip install tensorflow pandas numpy tqdm

In [25]:
import random
from collections import deque
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model, Sequential, load_model, clone_model
from tensorflow.keras.layers import Input, Add, Lambda, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import Huber

class CustomAgent:
    def __init__(self, state_size, strategy="t-dqn", reset_every=100, pretrained=False, model_name="./custom_model/custom_model.h5"):
        self.strategy = strategy
        self.state_size = state_size + 11
        self.action_size = 3
        self.model_name = model_name
        self.inventory = []
        self.memory = deque(maxlen=64)
        self.first_iter = True
        self.gamma = 0.95
        self.epsilon = 1.0
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995
        self.learning_rate = 0.001
        self.loss = Huber(delta=1.0)
        self.custom_objects = {"huber_loss": Huber(delta=1.0)}
        self.optimizer = Adam(learning_rate=self.learning_rate)

        if pretrained and self.model_name is not None:
            self.model = self.load_model()
        else:
            self.model = self.build_model()

        if self.strategy in ["t-dqn", "double-dqn", 'dueling-dqn']:
            self.n_iter = 1
            self.reset_every = reset_every
            self.target_model = clone_model(self.model)
            self.target_model.set_weights(self.model.get_weights())

    def build_model(self):
        model = Sequential()
        model.add(Dense(units=16, activation="relu", input_dim=self.state_size))
        model.add(Dense(units=32, activation="relu"))
        # model.add(Dense(units=32, activation="relu"))
        model.add(Dense(units=16, activation="relu"))
        model.add(Dense(units=self.action_size))
        model.compile(loss=self.loss, optimizer=self.optimizer)
        return model

    def train_experience_replay(self, batch_size):
        mini_batch = random.sample(self.memory, batch_size)
        X_train, y_train = [], []

        if self.strategy == "dqn":
            for state, action, reward, next_state, done in mini_batch:
                target = reward if done else reward + self.gamma * np.amax(self.model.predict(next_state)[0])
                q_values = self.model.predict(state)
                q_values[0][action] = target
                X_train.append(state[0])
                y_train.append(q_values[0])

        elif self.strategy == "t-dqn":
            for state, action, reward, next_state, done in mini_batch:
                target = reward if done else reward + self.gamma * np.amax(self.target_model.predict(next_state)[0])
                q_values = self.model.predict(state)
                q_values[0][action] = target
                X_train.append(state[0])
                y_train.append(q_values[0])

        elif self.strategy == "double-dqn":
            if self.n_iter % self.reset_every == 0:
                self.target_model.set_weights(self.model.get_weights())

            for state, action, reward, next_state, done in mini_batch:
                target = reward if done else reward + self.gamma * self.target_model.predict(next_state)[0][np.argmax(self.model.predict(next_state)[0])]
                q_values = self.model.predict(state)
                q_values[0][action] = target
                X_train.append(state[0])
                y_train.append(q_values[0])

        elif self.strategy == "dueling-dqn":
            if self.n_iter % self.reset_every == 0:
                self.target_model.set_weights(self.model.get_weights())

            for state, action, reward, next_state, done in mini_batch:
                next_q_values = self.target_model.predict(next_state)
                best_actions = np.argmax(next_q_values, axis=1)
                next_best_state_values = self.model.predict(next_state)[np.arange(batch_size), best_actions]
                target = reward + self.gamma * next_best_state_values

                q_values = self.model.predict(state)
                q_values[0][action] = target

                X_train.append(state[0])
                y_train.append(q_values[0])

        else:
            raise NotImplementedError()

        loss = self.model.fit(
            np.array(X_train), np.array(y_train),
            epochs=1, verbose=0
        ).history["loss"][0]

        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

        return loss

    def take_action(self, state, is_eval=False):
        if not is_eval and random.random() <= self.epsilon:
            return random.randrange(self.action_size)

        if self.first_iter:
            self.first_iter = False
            return 1

        action_probs = self.model.predict(state)
        return np.argmax(action_probs[0])

    def store_memory(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    def save_model(self, episode):
        self.model.save("{}_{}_{}.h5".format(self.model_name, self.state_size, episode))

    def load_model(self):
        return load_model(self.model_name, custom_objects=self.custom_objects)


In [26]:
import talib
def indicator(df):
    df['ADX'] = talib.ADX(df['High'], df['Low'], df['Close'], timeperiod=12)
    df['RSI'] = talib.RSI(df['Close'], timeperiod=12)
    df['SMA'] = talib.SMA(df['Close'], timeperiod=14)
    df['EMA'] = talib.EMA(df['Close'], timeperiod=14)
    df['WILLR'] = talib.WILLR(df['High'], df['Low'], df['Close'], timeperiod=12)
    #bollinger band
    df['BBANDS_upper'], df['BBANDS_middle'], df['BBANDS_lower'] = talib.BBANDS(df['Close'], timeperiod=12, nbdevup=2, nbdevdn=2, matype=0)
    #volume indicator
    df['VMA'] = talib.MOM(df['Volume'], timeperiod=12)
    # stochastic oscillator
    df['STOCH_slowk'], df['STOCH_slowd'] = talib.STOCH(df['High'], df['Low'], df['Close'], fastk_period=5, slowk_period=3, slowk_matype=0, slowd_period=3, slowd_matype=0)
    df.loc[:11, 'RSI'] = df['RSI'][12]
    df.loc[:11, 'ADX'] = df['ADX'][12]
    df.loc[:11, 'SMA'] = df['SMA'][12]
    df.loc[:11, 'EMA'] = df['EMA'][12]
    df.loc[:11, 'WILLR'] = df['WILLR'][12]
    df.loc[:11, 'BBANDS_upper'] = df['BBANDS_upper'][12]
    df.loc[:11, 'BBANDS_middle'] = df['BBANDS_middle'][12]
    df.loc[:11, 'BBANDS_lower'] = df['BBANDS_lower'][12]
    df.loc[:11, 'VMA'] = df['VMA'][12]
    df.loc[:11, 'STOCH_slowk'] = df['STOCH_slowk'][12]
    df.loc[:11, 'STOCH_slowd'] = df['STOCH_slowd'][12]

    return df

In [29]:
import math
import logging
import pandas as pd
import numpy as np
import talib
logging.basicConfig(level=logging.DEBUG)  # Set the logging level to DEBUG
format_pos = lambda price: ('-$' if price < 0 else '+$') + '{0:.2f}'.format(abs(price))
format_curr = lambda price: '${0:.2f}'.format(abs(price))

def display_train_result(result, val_pos, initial_offset):
    if val_pos == initial_offset or val_pos == 0.0:
        print('Episode {}/{} - Train Pos: USELESS  Val Pos: USELESS  Train Loss: {:.4f}'.format(result[0], result[1], result[3]))
        # logging.info('Episode {}/{} - Train Pos: {}  Val Pos: USELESS  Train Loss: {:.4f}'
        #              .format(result[0], result[1], format_pos(result[2]), result[3]))
    else:
        print('Episode {}/{} - Train Pos: {}  Val Pos: {}  Train Loss: {:.4f}'.format(result[0], result[1], format_pos(result[2]), format_pos(val_pos), result[3]))
        # logging.info('Episode {}/{} - Train Pos: {}  Val Pos: {}  Train Loss: {:.4f})'
        #              .format(result[0], result[1], format_pos(result[2]), format_pos(val_pos), result[3],))

def display_eval_result(model_name, profit, initial_offset):
    if profit == initial_offset or profit == 0.0:
        print('{}: USELESS\n'.format(model_name))
        # logging.info('{}: USELESS\n'.format(model_name))
    else:
        print('{}: {}\n'.format(model_name, format_pos(profit)))
        # logging.info('{}: {}\n'.format(model_name, format_pos(profit)))

def data_sample(df, sample_interval=2):
    df['Date'] = pd.to_datetime(df['Date'])
    df.set_index('Date', inplace=True)
    df_sampled = df.resample(str(sample_interval)+'D').last()
    df_sampled = df_sampled.reset_index()
    return df_sampled

def read_stock_data(stock_file):
    df = pd.read_csv(stock_file)
    print("Loaded Data Size - {}".format(df.shape))
    df = data_sample(df)
    print("Sampled Data Size - {}".format(df.shape))
    return df

def sigmoid_activation(x):
    try:
        if x < 0:
            return 1 - 1 / (1 + math.exp(x))
        return 1 / (1 + math.exp(-x))
    except Exception as e:
        print("Error in sigmoid: " + str(e))

def get_state_representation(df, t, n_days, indicator):

    data = list(df['Adj Close'])
    d = t - n_days + 1
    block = data[d: t + 1] if d >= 0 else -d * [data[0]] + data[0: t + 1]
    res = []
    for i in range(n_days - 1):
        res.append(sigmoid_activation(block[i + 1] - block[i]))
    rsi = indicator['RSI'][d + n_days - 1]
    adx = indicator['ADX'][d + n_days - 1]
    sma = indicator['SMA'][d + n_days - 1]
    ema = indicator['EMA'][d + n_days - 1]
    willr = indicator['WILLR'][d + n_days - 1]
    BBANDS_upper = indicator['BBANDS_upper'][d + n_days - 1]
    BBANDS_middle = indicator['BBANDS_middle'][d + n_days - 1]
    BBANDS_lower = indicator['BBANDS_lower'][d + n_days - 1]
    VMA = indicator['VMA'][d + n_days - 1]
    STOCH_slowk = indicator['STOCH_slowk'][d + n_days - 1]
    STOCH_slowd = indicator['STOCH_slowd'][d + n_days - 1]

    res.append(rsi)
    res.append(adx)
    res.append(sma)
    res.append(ema)
    res.append(willr)
    res.append(BBANDS_upper)
    res.append(BBANDS_middle)
    res.append(BBANDS_lower)
    res.append(VMA)
    res.append(STOCH_slowk)
    res.append(STOCH_slowd)
    return np.array([res])


In [30]:
import logging
import numpy as np
from tqdm import tqdm

logging.basicConfig(level=logging.DEBUG)  # Set the logging level to DEBUG
def train_custom_model(agent, episode, df, indicator, ep_count=20, batch_size=32, window_size=10):
    total_profit = 0
    data_length = len(df) - 1

    agent.inventory = []
    avg_loss = []

    state = get_state_representation(df, 0, window_size + 1, indicator)

    for t in tqdm(range(data_length), total=data_length, leave=True, desc=f'Episode {episode}/{ep_count}'):
        reward = 0
        next_state = get_state_representation(df, t + 1, window_size + 1, indicator)

        action = agent.take_action(state)

        if action == 1:
            agent.inventory.append(df['Adj Close'][t])

        elif action == 2 and len(agent.inventory) > 0:
            bought_price = agent.inventory.pop(0)
            delta = df['Adj Close'][t] - bought_price
            reward = delta
            total_profit += delta

        else:
            pass

        done = (t == data_length - 1)
        agent.store_memory(state, action, reward, next_state, done)

        if len(agent.memory) > batch_size:
            loss = agent.train_experience_replay(batch_size)
            avg_loss.append(loss)

        state = next_state

    agent.save_model(episode)

    return episode, ep_count, total_profit, np.mean(np.array(avg_loss))


def evaluate_custom_model(agent, df, indicator, window_size, debug):
    total_profit = 0
    data_length = len(df) - 1

    history = []
    agent.inventory = []
    data = list(df['Adj Close'])
    state = get_state_representation(df, 0, window_size + 1, indicator)

    for t in range(data_length):
        reward = 0
        next_state = get_state_representation(df, t + 1, window_size + 1, indicator)

        action = agent.take_action(state, is_eval=True)

        if action == 1:
            agent.inventory.append(data[t])
            # agent.inventory.append((df['Adj Close'][t],t))
            history.append((t, "BUY"))
            if debug:
                print("Buy at: {}".format(format_curr(data[t])))
                # logging.debug("Buy at: {}".format(format_curr(df['Adj Close'][t])))

        elif action == 2 and len(agent.inventory) > 0:
            bought_price = agent.inventory.pop(0)
            # bought_price,t = agent.inventory.pop(0)
            delta = df['Adj Close'][t] - bought_price
            reward = delta
            total_profit += delta
            history.append((t, "SELL"))
            if debug:
                print("Sell at: {} | Position: {}".format(format_curr(data[t]), format_pos(data[t] - bought_price)))
                # logging.debug("Sell at: {} | Position: {}".format(
                #     format_curr(df['Adj Close'][t]), format_pos(df['Adj Close'][t] - bought_price)))

        else:
            history.append((t, "HOLD"))

        done = (t == data_length - 1)
        agent.memory.append((state, action, reward, next_state, done))

        state = next_state
        if done:
            for i in range(len(agent.inventory)):
                total_profit += df['Adj Close'][t] - agent.inventory.pop(0)
            print('Profit:  {}'.format(format_pos(total_profit)))
            return total_profit, history

In [ ]:
def main(train_stock, val_stock, window_size, batch_size, ep_count,
         strategy="t-dqn", model_name="model_debug", pretrained=False,
         debug=False):

    custom_agent = CustomAgent(state_size=window_size, strategy=strategy, pretrained=pretrained, model_name=model_name)

    train_data = read_stock_data(train_stock)
    val_data = read_stock_data(val_stock)

    initial_offset = val_data['Adj Close'][1] - val_data['Adj Close'][0]
    indi_train = indicator(train_data)
    indi_val = indicator(val_data)
    for episode in range(1, ep_count + 1):
        train_result = train_custom_model(custom_agent, episode, train_data, indicator=indi_train, ep_count=ep_count,
                                   batch_size=batch_size, window_size=window_size)
        val_result, _ = evaluate_custom_model(custom_agent, val_data, indicator=indi_val, window_size=window_size, debug=debug)
        display_train_result(train_result, val_result, initial_offset)


if __name__ == "__main__":

    train_stock = '/content/TSLA_TRAIN.csv'
    val_stock = '/content/TSLA_VAL.csv'
    window_size = 12
    batch_size = 32
    ep_count = 15
    strategy = "double-dqn"
    model_name = "./model/TSLA_double_dqn"
    pretrained = False
    debug = False

    try:
        main(train_stock, val_stock, window_size, batch_size,
             ep_count, strategy=strategy, model_name=model_name,
             pretrained=pretrained, debug=debug)
    except KeyboardInterrupt:
        print("Aborted!")


1/1 [==============================] - 0s 27ms/step


Episode 1/15:  48%|████▊     | 249/524 [20:09<26:28,  5.78s/it]

1/1 [==============================] - 0s 25ms/step


Episode 1/15:  48%|████▊     | 250/524 [20:14<25:23,  5.56s/it]

1/1 [==============================] - 0s 25ms/step


...

1/1 [==============================] - 0s 45ms/step


Episode 1/15:  61%|██████▏   | 322/524 [27:08<20:40,  6.14s/it]

1/1 [==============================] - 0s 35ms/step


Episode 1/15:  62%|██████▏   | 323/524 [27:14<20:04,  5.99s/it]

1/1 [==============================] - 0s 32ms/step


In [ ]:
# Testing
import os

def main(eval_stock, window_size, model_name, debug):

    data = read_stock_data(eval_stock)
    initial_offset = data['Adj Close'][1] - data['Adj Close'][0]
    indic = indicator(data)
    # Single Model Evaluation
    if model_name is not None:
        custom_agent = CustomAgent(window_size, pretrained=True, model_name=model_name)
        profit, _ = evaluate_custom_model(custom_agent, data, indic, window_size, debug)
        display_eval_result(model_name, profit, initial_offset)

    # Multiple Model Evaluation
    else:
        for model in os.listdir("models"):
            if os.path.isfile(os.path.join("models", model)):
                custom_agent = CustomAgent(window_size, pretrained=True, model_name=model)
                profit = evaluate_custom_model(custom_agent, data, window_size, debug)
                display_eval_result(model, profit, initial_offset)
                del custom_agent


if __name__ == "__main__":

    eval_stock = '/kaggle/input/teslaa/TSLA_TEST.csv'
    window_size = 12
    model_name = '/kaggle/input/models23/TSLA_double_dqn_23_2.h5'
    debug = True
    try:
        main(eval_stock, window_size, model_name, debug)
    except KeyboardInterrupt:
        print("Aborted")


In [ ]:
df

,Date,Open,High,Low,Close,Adj Close,Volume
0,2010-08-12,35.950001,35.982857,35.584286,35.585712,23.913574,88717300
1,2010-08-16,36.119999,36.211430,35.525715,35.697144,23.988459,106676500
2,2010-08-20,35.970001,36.000000,35.035713,35.114285,23.596777,103510400
3,2010-08-24,34.535713,34.658573,33.651428,34.517143,23.195499,137097800
4,2010-08-28,34.549999,34.937141,34.335712,34.728573,23.337580,105196700
...,...,...,...,...,...,...,...
634,2017-07-22,151.800003,153.839996,151.800003,152.740005,149.876404,18853900
635,2017-07-26,149.889999,150.229996,149.190002,149.500000,146.697144,17213700
636,2017-07-30,159.279999,159.750000,156.160004,157.139999,154.193909,69936800
637,2017-08-03,156.070007,157.399994,155.690002,156.389999,153.457947,20559900
